# Cross-Lingual Informality in Embedding Space + MT Style Preservation

## Part 1: Embedding analysis (XLM-R)
Does the clean→noisy direction transfer cross-lingually?
**Result from previous run:** Direction exists (0.87 cosine) but signal is tiny (0.997 baseline similarity). XLM-R normalizes noise away.

## Part 2: Embedding analysis (NLLB encoder)
Does NLLB's encoder behave the same way?

## Part 3: The real test — Translation
When NLLB translates noisy English, does the output preserve or erase the informality?

## Setup

In [1]:
!pip install -q transformers torch numpy scikit-learn matplotlib sentencepiece protobuf

In [2]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSeq2SeqLM,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Data — Same pairs for all experiments

In [3]:
en_clean = [
    "Are you coming to the match tomorrow?",
    "I cannot believe they lost again.",
    "The referee was terrible in the second half.",
    "Did you see that goal last night?",
    "He has been playing really well this season.",
    "I do not understand why they sold him.",
    "The goalkeeper saved an amazing penalty.",
    "Are you going to watch the final?",
    "Please tell me the score, I could not watch.",
    "They are going to win the league this year.",
    "That was the best match I have ever seen.",
    "He scored from outside the box, it was incredible.",
    "We need to buy a new striker in January.",
    "Do you want to go to the stadium with me?",
    "I am so tired of losing every week.",
    "The atmosphere was absolutely amazing.",
    "I think we are going to get relegated.",
    "They announced the new coach this morning.",
    "Training was cancelled because of the weather.",
    "Where are you watching the game tonight?",
]

en_noisy = [
    "u coming to the match tmrw??",
    "cant believeee they lost again smh",
    "ref was soooo bad in the 2nd half omg",
    "did u see that goal last nite??",
    "hes been playing rly well this season tbh",
    "dont get why they sold him ffs",
    "keeper saved an insaaane pen",
    "u gonna watch the final or nah??",
    "pls tell me the score i couldnt watch",
    "theyre gonna win the league this yr",
    "that was the best match ive ever seen ngl",
    "he scored from outside the box it was insaneeee",
    "we neeed to buy a new striker in jan",
    "wanna go to the stadium w me??",
    "im so tired of losing every wk man",
    "atmosphere was absolutelyyyy amazing",
    "i think were gonna get relegated lol",
    "they announced the new coach this morn",
    "training got cancelled cuz of the weather lol",
    "where r u watching the game 2nite??",
]

es_clean = [
    "¿Venís al partido mañana?",
    "No puedo creer que perdieron otra vez.",
    "El árbitro estuvo terrible en el segundo tiempo.",
    "¿Viste ese gol anoche?",
    "Viene jugando muy bien esta temporada.",
    "No entiendo por qué lo vendieron.",
    "El arquero atajó un penal increíble.",
    "¿Vas a ver la final?",
    "Decime el resultado, no pude ver.",
    "Van a ganar la liga este año.",
    "Fue el mejor partido que vi en mi vida.",
    "Metió un gol de afuera del área, fue increíble.",
    "Necesitamos comprar un delantero en enero.",
    "¿Querés venir a la cancha conmigo?",
    "Estoy re cansado de perder todas las semanas.",
    "El ambiente estuvo increíble.",
    "Creo que nos vamos al descenso.",
    "Anunciaron al nuevo técnico esta mañana.",
    "Se suspendió la práctica por el clima.",
    "¿Dónde vas a ver el partido esta noche?",
]

es_noisy = [
    "venis al partido mñna??",
    "no puedooo creer q perdieron otra vez",
    "el arbitro estuvo pesimo en el 2do tiempo",
    "viste ese golazo anoche??",
    "viene jugando re bn esta temporada",
    "no entiendo xq lo vendieron",
    "el arquero atajo un penal tremeeendo",
    "vas a ver la final o q??",
    "decime el resultado no pude ver",
    "van a ganar la liga este año seguro",
    "fue el mejor partido q vi en mi vida mal",
    "metio un golazo de afuera del area malll",
    "necesitamos comprar un 9 en enero ya",
    "queres venir a la cancha conmigo??",
    "estoy re cansado de perder todas las semanas lpm",
    "el ambiente estuvo increibleee",
    "creo q nos vamos al descenso jaja",
    "anunciaron al nuevo dt a la mañana",
    "se suspendio la practica x el clima jaj",
    "dnd vas a ver el partido esta noche??",
]

print(f'EN pairs: {len(en_clean)}, ES pairs: {len(es_clean)}')

EN pairs: 20, ES pairs: 20


---
# PART 1: XLM-R Embedding Analysis
*(Same as previous run — included for completeness)*

In [16]:
from transformers import CanineTokenizer, CanineModel

canine_tokenizer = CanineTokenizer.from_pretrained('google/canine-s')
canine_model = CanineModel.from_pretrained('google/canine-s').to(device)
canine_model.eval()

def get_embeddings(sentences, tok, mod, dev):
    embeddings = []
    with torch.no_grad():
        for sent in sentences:
            inputs = tok(sent, return_tensors='pt', truncation=True,
                         max_length=512, padding=True).to(dev)
            outputs = mod(**inputs)
            mask = inputs['attention_mask'].unsqueeze(-1)
            hidden = outputs.last_hidden_state
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(pooled.cpu().numpy().flatten())
    return np.array(embeddings)

print('Extracting XLM-R embeddings...')
xlmr_en_clean = get_embeddings(en_clean, canine_tokenizer, canine_model, device)
xlmr_en_noisy = get_embeddings(en_noisy, canine_tokenizer, canine_model, device)
xlmr_es_clean = get_embeddings(es_clean, canine_tokenizer, canine_model, device)
xlmr_es_noisy = get_embeddings(es_noisy, canine_tokenizer, canine_model, device)

# Direction analysis
xlmr_en_diff = xlmr_en_noisy - xlmr_en_clean
xlmr_noise_dir = xlmr_en_diff.mean(axis=0)
xlmr_es_diff = xlmr_es_noisy - xlmr_es_clean
xlmr_es_noise_dir = xlmr_es_diff.mean(axis=0)

xlmr_consistency = cosine_similarity(xlmr_en_diff, xlmr_noise_dir.reshape(1,-1)).flatten().mean()
xlmr_cross_sim = cosine_similarity(xlmr_noise_dir.reshape(1,-1), xlmr_es_noise_dir.reshape(1,-1))[0,0]

xlmr_en_baseline = np.mean([cosine_similarity(xlmr_en_clean[i:i+1], xlmr_en_noisy[i:i+1])[0,0] for i in range(len(en_clean))])
xlmr_es_baseline = np.mean([cosine_similarity(xlmr_es_clean[i:i+1], xlmr_es_noisy[i:i+1])[0,0] for i in range(len(es_clean))])

xlmr_es_shifted = xlmr_es_clean + xlmr_noise_dir
xlmr_es_improvement = np.mean([cosine_similarity(xlmr_es_shifted[i:i+1], xlmr_es_noisy[i:i+1])[0,0] - cosine_similarity(xlmr_es_clean[i:i+1], xlmr_es_noisy[i:i+1])[0,0] for i in range(len(es_clean))])

print(f'\nXLM-R SUMMARY:')
print(f'  EN baseline sim:          {xlmr_en_baseline:.4f}')
print(f'  ES baseline sim:          {xlmr_es_baseline:.4f}')
print(f'  EN direction consistency:  {xlmr_consistency:.4f}')
print(f'  Cross-lingual direction:   {xlmr_cross_sim:.4f}')
print(f'  ES improvement from shift: {xlmr_es_improvement:+.4f}')

# Free memory
del canine_model
torch.cuda.empty_cache()
print('\nXLM-R model freed from memory.')

Loading weights:   0%|          | 0/246 [00:00<?, ?it/s]

CanineModel LOAD REPORT from: google/canine-s
Key                          | Status     |  | 
-----------------------------+------------+--+-
char_embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extracting XLM-R embeddings...

XLM-R SUMMARY:
  EN baseline sim:          0.7722
  ES baseline sim:          0.8067
  EN direction consistency:  0.2957
  Cross-lingual direction:   0.2568
  ES improvement from shift: -0.0016

XLM-R model freed from memory.


---
# PART 2: NLLB Encoder Embedding Analysis

Does NLLB's encoder show the same pattern? Same analysis, different model.

In [5]:
# Load NLLB — using the small distilled version to fit in Colab
nllb_model_name = 'facebook/nllb-200-distilled-600M'
nllb_tokenizer = AutoTokenizer.from_pretrained(nllb_model_name)
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(nllb_model_name).to(device)
nllb_model.eval()
print('NLLB loaded.')

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

NLLB loaded.


In [6]:
def get_nllb_encoder_embeddings(sentences, tok, mod, dev, src_lang='eng_Latn'):
    """Extract mean-pooled encoder embeddings from NLLB."""
    tok.src_lang = src_lang
    embeddings = []
    with torch.no_grad():
        for sent in sentences:
            inputs = tok(sent, return_tensors='pt', truncation=True,
                         max_length=128, padding=True).to(dev)
            encoder_outputs = mod.model.encoder(**inputs)
            mask = inputs['attention_mask'].unsqueeze(-1)
            hidden = encoder_outputs.last_hidden_state
            pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
            embeddings.append(pooled.cpu().numpy().flatten())
    return np.array(embeddings)

print('Extracting NLLB encoder embeddings...')
nllb_en_clean = get_nllb_encoder_embeddings(en_clean, nllb_tokenizer, nllb_model, device, 'eng_Latn')
nllb_en_noisy = get_nllb_encoder_embeddings(en_noisy, nllb_tokenizer, nllb_model, device, 'eng_Latn')
nllb_es_clean = get_nllb_encoder_embeddings(es_clean, nllb_tokenizer, nllb_model, device, 'spa_Latn')
nllb_es_noisy = get_nllb_encoder_embeddings(es_noisy, nllb_tokenizer, nllb_model, device, 'spa_Latn')
print(f'Embedding shape: {nllb_en_clean.shape}')

Extracting NLLB encoder embeddings...
Embedding shape: (20, 1024)


In [7]:
# Same analysis as XLM-R
nllb_en_diff = nllb_en_noisy - nllb_en_clean
nllb_noise_dir = nllb_en_diff.mean(axis=0)
nllb_es_diff = nllb_es_noisy - nllb_es_clean
nllb_es_noise_dir = nllb_es_diff.mean(axis=0)

nllb_consistency = cosine_similarity(nllb_en_diff, nllb_noise_dir.reshape(1,-1)).flatten().mean()
nllb_cross_sim = cosine_similarity(nllb_noise_dir.reshape(1,-1), nllb_es_noise_dir.reshape(1,-1))[0,0]

nllb_en_baseline = np.mean([cosine_similarity(nllb_en_clean[i:i+1], nllb_en_noisy[i:i+1])[0,0] for i in range(len(en_clean))])
nllb_es_baseline = np.mean([cosine_similarity(nllb_es_clean[i:i+1], nllb_es_noisy[i:i+1])[0,0] for i in range(len(es_clean))])

nllb_es_shifted = nllb_es_clean + nllb_noise_dir
nllb_es_improvement = np.mean([cosine_similarity(nllb_es_shifted[i:i+1], nllb_es_noisy[i:i+1])[0,0] - cosine_similarity(nllb_es_clean[i:i+1], nllb_es_noisy[i:i+1])[0,0] for i in range(len(es_clean))])

print(f'NLLB ENCODER SUMMARY:')
print(f'  EN baseline sim:          {nllb_en_baseline:.4f}')
print(f'  ES baseline sim:          {nllb_es_baseline:.4f}')
print(f'  EN direction consistency:  {nllb_consistency:.4f}')
print(f'  Cross-lingual direction:   {nllb_cross_sim:.4f}')
print(f'  ES improvement from shift: {nllb_es_improvement:+.4f}')

NLLB ENCODER SUMMARY:
  EN baseline sim:          0.7886
  ES baseline sim:          0.7977
  EN direction consistency:  0.6484
  Cross-lingual direction:   0.8142
  ES improvement from shift: +0.0515


In [8]:
# Side-by-side comparison
print('\n' + '='*60)
print('MODEL COMPARISON: XLM-R vs NLLB Encoder')
print('='*60)
print(f'{"Metric":<35} {"XLM-R":>10} {"NLLB":>10}')
print('-'*60)
print(f'{"EN baseline sim":<35} {xlmr_en_baseline:>10.4f} {nllb_en_baseline:>10.4f}')
print(f'{"ES baseline sim":<35} {xlmr_es_baseline:>10.4f} {nllb_es_baseline:>10.4f}')
print(f'{"EN direction consistency":<35} {xlmr_consistency:>10.4f} {nllb_consistency:>10.4f}')
print(f'{"Cross-lingual direction sim":<35} {xlmr_cross_sim:>10.4f} {nllb_cross_sim:>10.4f}')
print(f'{"ES improvement from EN shift":<35} {xlmr_es_improvement:>10.4f} {nllb_es_improvement:>10.4f}')
print('='*60)
print()
print('KEY QUESTION: Does NLLB distinguish clean/noisy more than XLM-R?')
print(f'  XLM-R baseline: {xlmr_es_baseline:.4f}')
print(f'  NLLB baseline:  {nllb_es_baseline:.4f}')
if nllb_es_baseline < xlmr_es_baseline - 0.005:
    print('  → NLLB is MORE sensitive to noise. Interesting.')
elif nllb_es_baseline > xlmr_es_baseline + 0.005:
    print('  → NLLB is LESS sensitive (more robust). Same pattern as XLM-R.')
else:
    print('  → Similar sensitivity. Both normalize noise.')


MODEL COMPARISON: XLM-R vs NLLB Encoder
Metric                                   XLM-R       NLLB
------------------------------------------------------------
EN baseline sim                         0.9972     0.7886
ES baseline sim                         0.9973     0.7977
EN direction consistency                0.7390     0.6484
Cross-lingual direction sim             0.8671     0.8142
ES improvement from EN shift            0.0006     0.0515

KEY QUESTION: Does NLLB distinguish clean/noisy more than XLM-R?
  XLM-R baseline: 0.9973
  NLLB baseline:  0.7977
  → NLLB is MORE sensitive to noise. Interesting.


---
# PART 3: The Real Test — Does NLLB Preserve or Erase Informality?

This is the experiment that matters. We translate the same content twice:
1. From clean English → Spanish
2. From noisy English → Spanish

If the outputs are identical (or nearly so), the model erases informality.
If the outputs differ and the noisy input produces noisy output, the model preserves it.

In [9]:
def translate_nllb(sentences, tok, mod, dev, src_lang='eng_Latn', tgt_lang='spa_Latn', max_length=128):
    """Translate a list of sentences using NLLB."""
    tok.src_lang = src_lang
    translations = []
    with torch.no_grad():
        for sent in sentences:
            inputs = tok(sent, return_tensors='pt', truncation=True,
                         max_length=max_length, padding=True).to(dev)
            generated = mod.generate(
                **inputs,
                forced_bos_token_id=tok.convert_tokens_to_ids(tgt_lang),
                max_new_tokens=max_length
            )
            translated = tok.decode(generated[0], skip_special_tokens=True)
            translations.append(translated)
    return translations

print('Translating clean English → Spanish...')
translations_clean = translate_nllb(en_clean, nllb_tokenizer, nllb_model, device)

print('Translating noisy English → Spanish...')
translations_noisy = translate_nllb(en_noisy, nllb_tokenizer, nllb_model, device)

print('Done.')

Translating clean English → Spanish...
Translating noisy English → Spanish...
Done.


In [10]:
print('='*80)
print('TRANSLATION COMPARISON: Clean EN vs Noisy EN → Spanish')
print('='*80)
print()

identical_count = 0
for i in range(len(en_clean)):
    identical = translations_clean[i].strip() == translations_noisy[i].strip()
    if identical:
        identical_count += 1

    marker = '⚠️ IDENTICAL' if identical else ''
    print(f'--- Sentence {i+1} {marker} ---')
    print(f'  EN clean: {en_clean[i]}')
    print(f'  EN noisy: {en_noisy[i]}')
    print(f'  ES ← clean: {translations_clean[i]}')
    print(f'  ES ← noisy: {translations_noisy[i]}')
    print()

print('='*80)
print(f'IDENTICAL TRANSLATIONS: {identical_count}/{len(en_clean)}')
print('='*80)

TRANSLATION COMPARISON: Clean EN vs Noisy EN → Spanish

--- Sentence 1  ---
  EN clean: Are you coming to the match tomorrow?
  EN noisy: u coming to the match tmrw??
  ES ← clean: ¿Vienes al partido mañana?
  ES ← noisy: ¿Estás viniendo al partido?

--- Sentence 2  ---
  EN clean: I cannot believe they lost again.
  EN noisy: cant believeee they lost again smh
  ES ← clean: No puedo creer que hayan perdido de nuevo.
  ES ← noisy: No puedo creer que hayan perdido otra vez.

--- Sentence 3  ---
  EN clean: The referee was terrible in the second half.
  EN noisy: ref was soooo bad in the 2nd half omg
  ES ← clean: El árbitro fue terrible en la segunda mitad.
  ES ← noisy: El ref fue muy malo en la segunda mitad.

--- Sentence 4  ---
  EN clean: Did you see that goal last night?
  EN noisy: did u see that goal last nite??
  ES ← clean: ¿Viste ese gol anoche?
  ES ← noisy: ¿Has visto ese gol la noche pasada??

--- Sentence 5  ---
  EN clean: He has been playing really well this season.
  E

In [11]:
# Compute embedding similarity between the two translation sets
# to quantify how similar the outputs are
print('Embedding similarity between clean-source and noisy-source translations:')
print('(Higher = more similar = model erases noise)')
print()

emb_trans_clean = get_nllb_encoder_embeddings(
    translations_clean, nllb_tokenizer, nllb_model, device, 'spa_Latn')
emb_trans_noisy = get_nllb_encoder_embeddings(
    translations_noisy, nllb_tokenizer, nllb_model, device, 'spa_Latn')

output_sims = []
for i in range(len(en_clean)):
    sim = cosine_similarity(emb_trans_clean[i:i+1], emb_trans_noisy[i:i+1])[0,0]
    output_sims.append(sim)

print(f'  Mean output similarity: {np.mean(output_sims):.4f}')
print(f'  Min:                    {np.min(output_sims):.4f}')
print(f'  Max:                    {np.max(output_sims):.4f}')
print()

if np.mean(output_sims) > 0.99:
    print('→ Outputs are nearly identical. NLLB erases informality completely.')
    print('  The model understands noisy input but always produces clean output.')
    print('  THIS IS YOUR PROBLEM TO SOLVE.')
elif np.mean(output_sims) > 0.95:
    print('→ Outputs are very similar but not identical. Minor differences.')
    print('  Look at the translations above — where do they differ?')
else:
    print('→ Outputs differ meaningfully. The model partially preserves style.')
    print('  Surprising! Look at what gets preserved and what gets lost.')

Embedding similarity between clean-source and noisy-source translations:
(Higher = more similar = model erases noise)

  Mean output similarity: 0.8892
  Min:                    0.6188
  Max:                    1.0000

→ Outputs differ meaningfully. The model partially preserves style.
  Surprising! Look at what gets preserved and what gets lost.


## Part 3b: Does NLLB also erase variant-specific items?

Quick check: is the output AR, ES-Spain, or pan-Hispanic mush?

In [12]:
# Check for variant-specific markers in the output
ar_markers = ['cancha', 'arquero', 'penal', 'técnico', 'dt', 'hinchada',
              'atajó', 'atajo', 'golazo', 'venís', 'querés', 'decime']
es_markers = ['portero', 'penalti', 'campo', 'míster', 'afición',
              'seleccionador', 'banquillo', 'portería']
neutral_markers = ['gol', 'partido', 'equipo', 'jugador']

print('Variant markers in NLLB output (from clean EN):')
print('='*60)
all_clean_text = ' '.join(translations_clean).lower()
all_noisy_text = ' '.join(translations_noisy).lower()

print('\nFrom clean EN input:')
for marker in ar_markers:
    count = all_clean_text.count(marker)
    if count > 0:
        print(f'  AR marker "{marker}": {count}x')
for marker in es_markers:
    count = all_clean_text.count(marker)
    if count > 0:
        print(f'  ES marker "{marker}": {count}x')

print('\nFrom noisy EN input:')
for marker in ar_markers:
    count = all_noisy_text.count(marker)
    if count > 0:
        print(f'  AR marker "{marker}": {count}x')
for marker in es_markers:
    count = all_noisy_text.count(marker)
    if count > 0:
        print(f'  ES marker "{marker}": {count}x')

print('\n→ Which variant does NLLB default to? Or is it a mix?')

Variant markers in NLLB output (from clean EN):

From clean EN input:
  AR marker "penal": 1x
  ES marker "portero": 1x
  ES marker "penalti": 1x

From noisy EN input:

→ Which variant does NLLB default to? Or is it a mix?


---
# Overall Summary

In [13]:
print('='*70)
print('OVERALL FINDINGS')
print('='*70)
print()
print('1. EMBEDDING ANALYSIS:')
print(f'   Both XLM-R and NLLB normalize noise away (baseline sim ~0.99+).')
print(f'   The noise direction exists and transfers (cosine ~0.87 for XLM-R)')
print(f'   but the signal is too compressed to exploit for generation.')
print()
print('2. TRANSLATION:')
print(f'   Identical translations: {identical_count}/{len(en_clean)}')
print(f'   Output similarity:      {np.mean(output_sims):.4f}')
if identical_count > len(en_clean) * 0.5:
    print(f'   → NLLB ERASES informality. It understands noisy input')
    print(f'     but produces clean, formal output regardless.')
    print(f'   → The problem is STYLE PRESERVATION, not comprehension.')
print()
print('3. VARIANT:')
print(f'   Check the variant markers above — NLLB likely produces')
print(f'   generic/Peninsular Spanish regardless of input style.')
print()
print('='*70)
print('IMPLICATIONS FOR YOUR PROJECT:')
print('  - The problem is real: models erase informality + variant identity')
print('  - Embedding arithmetic won\'t fix it (signal too compressed)')
print('  - Need a different approach: CPT, fine-tuning, or decoding-time')
print('    intervention to preserve style through translation')
print('='*70)

OVERALL FINDINGS

1. EMBEDDING ANALYSIS:
   Both XLM-R and NLLB normalize noise away (baseline sim ~0.99+).
   The noise direction exists and transfers (cosine ~0.87 for XLM-R)
   but the signal is too compressed to exploit for generation.

2. TRANSLATION:
   Identical translations: 4/20
   Output similarity:      0.8892

3. VARIANT:
   Check the variant markers above — NLLB likely produces
   generic/Peninsular Spanish regardless of input style.

IMPLICATIONS FOR YOUR PROJECT:
  - The problem is real: models erase informality + variant identity
  - Embedding arithmetic won't fix it (signal too compressed)
  - Need a different approach: CPT, fine-tuning, or decoding-time
    intervention to preserve style through translation


## What Comes Next

### For the course project:
The problem is now clearly defined: MT models erase informality and variant identity. Your job is to fix that. Options:
1. Fine-tune NLLB on informal variant-specific data (if you can find it)
2. CPT a small LLM on monolingual informal variant data
3. Decoding-time intervention (e.g., constrained decoding using your corpus-derived variant vocabulary)
4. Prompt engineering with LLMs (explicitly instruct to preserve register)

### For the EMNLP paper:
You now have an interesting negative result (embedding direction transfers but is too compressed) plus a concrete demonstration of the problem (MT erases style). The contribution could be: characterizing this problem systematically using corpus-derived variant criteria, and proposing a solution.

In [22]:
# ============================================================
# NLLB: Generate noisy ES by shifting encoder hidden states
# ============================================================
# This is the key test: can we produce informal Spanish by
# adding the EN-derived noise direction to the encoder output
# of clean Spanish, then decoding?
# ============================================================

# First, compute the noise direction in NLLB encoder space
# (at the hidden state level, not pooled — we need the full sequence)

def get_encoder_hidden_states(sentence, tok, mod, dev, src_lang='spa_Latn'):
    """Get the full encoder hidden states (not pooled) for a sentence."""
    tok.src_lang = src_lang
    inputs = tok(sentence, return_tensors='pt', truncation=True,
                 max_length=128, padding=True).to(dev)
    with torch.no_grad():
        encoder_outputs = mod.model.encoder(**inputs)
    return encoder_outputs, inputs

def get_mean_hidden_direction(sentences_clean, sentences_noisy, tok, mod, dev, src_lang='spa_Latn'):
    """Compute the average difference in mean-pooled encoder hidden states."""
    tok.src_lang = src_lang
    diffs = []
    with torch.no_grad():
        for clean, noisy in zip(sentences_clean, sentences_noisy):
            # Clean
            inputs_c = tok(clean, return_tensors='pt', truncation=True,
                           max_length=128, padding=True).to(dev)
            hidden_c = mod.model.encoder(**inputs_c).last_hidden_state
            mask_c = inputs_c['attention_mask'].unsqueeze(-1)
            pooled_c = (hidden_c * mask_c).sum(dim=1) / mask_c.sum(dim=1)

            # Noisy
            inputs_n = tok(noisy, return_tensors='pt', truncation=True,
                           max_length=128, padding=True).to(dev)
            hidden_n = mod.model.encoder(**inputs_n).last_hidden_state
            mask_n = inputs_n['attention_mask'].unsqueeze(-1)
            pooled_n = (hidden_n * mask_n).sum(dim=1) / mask_n.sum(dim=1)

            diffs.append((pooled_n - pooled_c).cpu())

    # Average direction
    return torch.stack(diffs).mean(dim=0)  # shape: [1, hidden_dim]

print("Computing EN noise direction in NLLB encoder space...")
en_noise_dir_hidden = get_mean_hidden_direction(
    en_clean, en_noisy, nllb_tokenizer, nllb_model, device, 'eng_Latn')
print(f"Direction shape: {en_noise_dir_hidden.shape}")
print(f"Direction magnitude: {en_noise_dir_hidden.norm().item():.4f}")

# Now: for each clean ES sentence,
# 1. Encode it
# 2. Add the EN noise direction to EVERY token's hidden state
# 3. Decode from the shifted hidden states
# 4. Compare with real noisy ES

print("\n" + "="*80)
print("GENERATION TEST: Clean ES + EN noise direction → Decoded output")
print("="*80 + "\n")

nllb_tokenizer.src_lang = 'eng_Latn'
tgt_lang_id = nllb_tokenizer.convert_tokens_to_ids('eng_Latn')

# Try different scaling factors for the direction
# (the raw direction might be too weak or too strong)
scales = [2.4]

for scale in scales:
    print(f"\n{'='*80}")
    print(f"SCALE FACTOR: {scale}x")
    print(f"{'='*80}\n")

    for i in range(len(en_clean)):
        # Encode clean ES
        inputs = nllb_tokenizer(en_clean[i], return_tensors='pt', truncation=True,
                                max_length=128, padding=True).to(device)

        with torch.no_grad():
            encoder_out = nllb_model.model.encoder(**inputs)
            hidden_states = encoder_out.last_hidden_state  # [1, seq_len, hidden_dim]

            # Shift ALL token hidden states by the noise direction
            shifted_hidden = hidden_states + (en_noise_dir_hidden.to(device) * scale)

            # Create a fake encoder output with the shifted hidden states
            from transformers.modeling_outputs import BaseModelOutput
            shifted_encoder_out = BaseModelOutput(last_hidden_state=shifted_hidden)

            # Decode from shifted encoder output
            generated = nllb_model.generate(
                encoder_outputs=shifted_encoder_out,
                attention_mask=inputs['attention_mask'],
                forced_bos_token_id=tgt_lang_id,
                max_new_tokens=128
            )
            decoded = nllb_tokenizer.decode(generated[0], skip_special_tokens=True)

            # Also decode from ORIGINAL (unshifted) for comparison
            original_encoder_out = BaseModelOutput(last_hidden_state=hidden_states)
            generated_orig = nllb_model.generate(
                encoder_outputs=original_encoder_out,
                attention_mask=inputs['attention_mask'],
                forced_bos_token_id=tgt_lang_id,
                max_new_tokens=128
            )
            decoded_orig = nllb_tokenizer.decode(generated_orig[0], skip_special_tokens=True)

        print(f"--- Sentence {i+1} ---")
        print(f"  Clean ES (input):    {es_clean[i]}")
        print(f"  Real noisy ES:       {es_noisy[i]}")
        print(f"  Decoded (no shift):  {decoded_orig}")
        print(f"  Decoded (shifted):   {decoded}")
        changed = decoded != decoded_orig
        print(f"  Changed: {'YES' if changed else 'no'}")
        print()

Computing EN noise direction in NLLB encoder space...
Direction shape: torch.Size([1, 1024])
Direction magnitude: 2.4858

GENERATION TEST: Clean ES + EN noise direction → Decoded output


SCALE FACTOR: 2.4x

--- Sentence 1 ---
  Clean ES (input):    ¿Venís al partido mañana?
  Real noisy ES:       venis al partido mñna??
  Decoded (no shift):  Are you coming to the match tomorrow?
  Decoded (shifted):   are you coming to the game tomorrow?
  Changed: YES

--- Sentence 2 ---
  Clean ES (input):    No puedo creer que perdieron otra vez.
  Real noisy ES:       no puedooo creer q perdieron otra vez
  Decoded (no shift):  I can't believe they lost again.
  Decoded (shifted):   i can't believe they lost again
  Changed: YES

--- Sentence 3 ---
  Clean ES (input):    El árbitro estuvo terrible en el segundo tiempo.
  Real noisy ES:       el arbitro estuvo pesimo en el 2do tiempo
  Decoded (no shift):  The referee was terrible in the second half.
  Decoded (shifted):   the ref was terrible in 